# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:**: Solucionar problema de calidad encontrados en el EDA: consistencia, sensibilidad, precision y completitud. Documenta cada decision tomada.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numerica, ya que los algoritmos clasicos no entienen el texto. Documenta porque usaste codificacion Ordinal o Nominal.
3. **Enriquecimiento (Feature Engineering):** Como pudiste notar en tu análisis, `total_rooms` no significa mucho si hay muchos hogares en un distrito. Agrega nuevas métricas útiles, por ejemplo:
   - `rooms_per_household = total_rooms / households`
   - `bedrooms_per_room = total_bedrooms / total_rooms`
   - `population_per_household = population / households`
4. **Escalado de Variables:** Aplica un `StandardScaler` o `MinMaxScaler` para evitar que las variables numéricas grandes pesen más en algoritmos basados en distancias o gradientes.


In [ ]:
# =========================================================
# 02. LIMPIEZA Y ENRIQUECIMIENTO DE DATOS
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

# ---------------------------------------------------------
# 1. Partimos de lo ya hecho en la secuencia:
#    train ya fue cargado en el notebook anterior o aquí.
# ---------------------------------------------------------

train = pd.read_csv("../data/interim/train_set.csv")

X = train.drop("median_house_value", axis=1).copy()
y = train["median_house_value"].copy()

print("Shape original de X:", X.shape)
print("Shape de y:", y.shape)


In [ ]:
# ---------------------------------------------------------
# 2. Diagnóstico rápido alineado al EDA
# ---------------------------------------------------------

print("\nValores faltantes:")
print(X.isnull().sum().sort_values(ascending=False))

print("\nDistribución de ocean_proximity:")
print(X["ocean_proximity"].value_counts(dropna=False))

# ---------------------------------------------------------
# 3. Feature engineering experto
# ---------------------------------------------------------

def add_advanced_features(df):
    df = df.copy()
    eps = 1e-6

    # =========================
    # Variables relativas base
    # =========================
    df["rooms_per_household"] = df["total_rooms"] / (df["households"] + eps)
    df["bedrooms_per_room"] = df["total_bedrooms"] / (df["total_rooms"] + eps)
    df["population_per_household"] = df["population"] / (df["households"] + eps)

    # =========================
    # Variables adicionales de mayor valor analítico
    # =========================
    df["bedrooms_per_household"] = df["total_bedrooms"] / (df["households"] + eps)
    df["rooms_per_person"] = df["total_rooms"] / (df["population"] + eps)
    df["income_per_household"] = df["median_income"] / (df["households"] + eps)
    df["income_per_room"] = df["median_income"] / (df["rooms_per_household"] + eps)

    # =========================
    # Interacciones geográficas
    # =========================
    df["geo_interaction"] = df["latitude"] * df["longitude"]
    df["distance_proxy"] = np.sqrt(df["latitude"]**2 + df["longitude"]**2)

    # =========================
    # Transformaciones log para reducir asimetría / outliers
    # =========================
    df["log_total_rooms"] = np.log1p(df["total_rooms"])
    df["log_total_bedrooms"] = np.log1p(df["total_bedrooms"])
    df["log_population"] = np.log1p(df["population"])
    df["log_households"] = np.log1p(df["households"])
    df["log_median_income"] = np.log1p(df["median_income"])

    return df

# ---------------------------------------------------------
# 4. Aplicar feature engineering para análisis intermedio
# ---------------------------------------------------------

X_fe = add_advanced_features(X)

print("\nShape después de feature engineering:", X_fe.shape)
print("\nNuevas columnas creadas:")
new_cols = [c for c in X_fe.columns if c not in X.columns]
print(new_cols)

# ---------------------------------------------------------
# 5. Revisar aporte de variables nuevas frente a la target
# ---------------------------------------------------------

tmp = X_fe.copy()
tmp["median_house_value"] = y

corr_target = tmp.corr(numeric_only=True)["median_house_value"].sort_values(ascending=False)

print("\nCorrelaciones con median_house_value:")
print(corr_target.head(20))

# ---------------------------------------------------------
# 6. Visualizaciones clave de variables derivadas
# ---------------------------------------------------------

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

sns.scatterplot(data=tmp, x="rooms_per_household", y="median_house_value", alpha=0.15, ax=axes[0, 0])
axes[0, 0].set_title("Rooms per household vs Price")

sns.scatterplot(data=tmp, x="bedrooms_per_room", y="median_house_value", alpha=0.15, ax=axes[0, 1])
axes[0, 1].set_title("Bedrooms per room vs Price")

sns.scatterplot(data=tmp, x="population_per_household", y="median_house_value", alpha=0.15, ax=axes[0, 2])
axes[0, 2].set_title("Population per household vs Price")

sns.scatterplot(data=tmp, x="income_per_household", y="median_house_value", alpha=0.15, ax=axes[1, 0])
axes[1, 0].set_title("Income per household vs Price")

sns.scatterplot(data=tmp, x="income_per_room", y="median_house_value", alpha=0.15, ax=axes[1, 1])
axes[1, 1].set_title("Income per room vs Price")

sns.scatterplot(data=tmp, x="log_median_income", y="median_house_value", alpha=0.15, ax=axes[1, 2])
axes[1, 2].set_title("Log median income vs Price")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 7. Definir variables numéricas y categóricas
# ---------------------------------------------------------

num_attribs = X_fe.drop(columns=["ocean_proximity"]).columns.tolist()
cat_attribs = ["ocean_proximity"]

print("\nCantidad de variables numéricas:", len(num_attribs))
print("Variables categóricas:", cat_attribs)

# ---------------------------------------------------------
# 8. Pipeline numérico
# ---------------------------------------------------------
# Decisión:
# - SimpleImputer con mediana: robusto ante outliers
# - StandardScaler: útil para algoritmos sensibles a escala
# ---------------------------------------------------------

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# ---------------------------------------------------------
# 9. Pipeline categórico
# ---------------------------------------------------------
# OneHotEncoder es correcto porque ocean_proximity es nominal,
# no ordinal.
# ---------------------------------------------------------

cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# ---------------------------------------------------------
# 10. Pipeline completo
# ---------------------------------------------------------

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs)
])

# ---------------------------------------------------------
# 11. Transformar dataset final
# ---------------------------------------------------------

X_prepared = full_pipeline.fit_transform(X_fe)

print("\nShape final transformado:", X_prepared.shape)

# ---------------------------------------------------------
# 12. Recuperar nombres de columnas finales
# ---------------------------------------------------------

ohe = full_pipeline.named_transformers_["cat"]["onehot"]
cat_feature_names = ohe.get_feature_names_out(cat_attribs)

final_feature_names = num_attribs + list(cat_feature_names)

X_prepared_df = pd.DataFrame(X_prepared, columns=final_feature_names)

print("\nPrimeras filas del dataset transformado:")
display(X_prepared_df.head())

print("\nValores faltantes finales en dataset transformado:", X_prepared_df.isnull().sum().sum())

# ---------------------------------------------------------
# 13. Chequeo resumido de escala
# ---------------------------------------------------------

print("\nResumen estadístico del dataset transformado:")
display(X_prepared_df.describe().T.head(20))